# Module 3 — Intent Classifier (Sakina)

**Brief requirement:** zero-shot *or* few-shot LLM prompting to classify intent into one of 5 labels (greeting, goodbye, gratitude, asking_mental_health_question, out_of_scope).

**Approach:** hand-written **few-shot** prompting to `gpt-oss-120b` (Lightning) — no DSPy. DSPy/MIPROv2 was tried and removed: it scored **95.83% vs 100% for zero-shot** on the held-out test (it overfit the 24-example dev set) while adding a ~33-min compile and a heavy dependency. We export `intent_few_shot.json` for the API — no `dspy` at runtime.

**Results:** held-out test (n=24) **100.0%**; 20-language probe (n=450) **92.0%** (the low *greeting* score is a probe artifact — many synthetic greetings are compound greeting + off-topic, reasonably labeled `out_of_scope`).

## 1. Setup + rate-limited LLM client

Lightning caps the API at **15 req/min, 120k tok/min** (even from a local machine), so every call goes through a sliding-window rate limiter.

In [ ]:
import os
import random
import threading
import time
from collections import deque
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACTS_DIR = Path("..") / "api" / "app" / "models" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

ENV_PATH = Path("../api/.env")
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f"loaded env from {ENV_PATH.resolve()}")
else:
    print(f"WARNING: {ENV_PATH} not found — set LIGHTNING_* env vars another way")

LIGHTNING_BASE_URL = os.environ["LIGHTNING_BASE_URL"]
LIGHTNING_API_KEY = os.environ["LIGHTNING_API_KEY"]
LIGHTNING_MODEL = os.environ.get("LIGHTNING_MODEL", "lightning-ai/gpt-oss-120b")
print(f"model: {LIGHTNING_MODEL}")


# Throttle every LM call: Lightning caps at 15 req/min, 120k tok/min (even from local).
class RateLimiter:
    def __init__(self, max_req_per_min=15, window_s=60.0, safety_margin=0.9):
        self.max_req = int(max_req_per_min * safety_margin)
        self.window_s = window_s
        self._events: deque[float] = deque()
        self._lock = threading.Lock()

    def acquire(self) -> None:
        while True:
            with self._lock:
                now = time.monotonic()
                cutoff = now - self.window_s
                while self._events and self._events[0] < cutoff:
                    self._events.popleft()
                if len(self._events) + 1 <= self.max_req:
                    self._events.append(now)
                    return
                wait = max(0.1, self._events[0] + self.window_s - now + 0.05)
            time.sleep(wait)


RATE_LIMITER = RateLimiter()

In [ ]:
import pandas as pd
from openai import OpenAI

# Plain OpenAI-compatible client (Lightning), no DSPy. DSPy/MIPROv2 was removed:
# it scored 95.83% vs 100% zero-shot here (overfit the 24-example dev set).
client = OpenAI(base_url=LIGHTNING_BASE_URL, api_key=LIGHTNING_API_KEY)
INTENT_MODEL = LIGHTNING_MODEL
INTENT_MAX_TOKENS = 512                 # 40 was too small for a reasoning model — it truncated the answer
print("intent model:", INTENT_MODEL, "| max_tokens:", INTENT_MAX_TOKENS)

## 2. Intent labels (fixed by brief)

In [ ]:
INTENTS = [
    "greeting",
    "goodbye",
    "gratitude",
    "asking_mental_health_question",
    "out_of_scope",
]

INTENT_DESCRIPTIONS = {
    "greeting": "hello / salutations / opening a conversation",
    "goodbye": "farewell / closing the conversation",
    "gratitude": "thank you / appreciation for help received",
    "asking_mental_health_question": "a question or statement about feelings, mental state, relationships, stress, anxiety, depression, sleep, or any topic Sakina can help with",
    "out_of_scope": "anything else — coding questions, weather, sports, math, current events, etc.",
}

for intent, desc in INTENT_DESCRIPTIONS.items():
    print(f"  {intent:<32} {desc}")

## 3. Hand-labeled dataset (~120 messages, EN + AR)

We pool the few-shot bank with the hand-labeled set and split train/dev/test, stratified by intent: demos come from **train**, **test** is the held-out report set. (The 20-language probe in §8 is a separate multilingual check.)

In [ ]:
LABELED = [
    # greeting
    ("Hi there", "greeting"),
    ("Hello, how are you?", "greeting"),
    ("السلام عليكم", "greeting"),
    ("أهلا", "greeting"),
    ("Hey Sakina", "greeting"),
    ("Good morning!", "greeting"),
    ("Hi, are you there?", "greeting"),
    ("yo", "greeting"),
    ("hello again", "greeting"),
    ("howdy", "greeting"),
    ("Greetings", "greeting"),
    ("good evening", "greeting"),
    ("hey", "greeting"),
    ("hiya", "greeting"),
    ("صباح الخير", "greeting"),
    ("مساء الخير يا سكينة", "greeting"),
    ("هاي", "greeting"),
    ("إزيك", "greeting"),
    ("عامل ايه؟", "greeting"),
    ("سلام", "greeting"),
    ("اهلا وسهلا", "greeting"),
    ("أهلين", "greeting"),
    ("يا هلا", "greeting"),
    ("صباحك سعيد", "greeting"),
    # goodbye
    ("Bye for now, talk later", "goodbye"),
    ("I have to go, take care", "goodbye"),
    ("مع السلامة", "goodbye"),
    ("تصبح على خير", "goodbye"),
    ("Goodbye", "goodbye"),
    ("see you later", "goodbye"),
    ("I gotta go now", "goodbye"),
    ("talk soon", "goodbye"),
    ("bye bye", "goodbye"),
    ("catch you later", "goodbye"),
    ("that's all for today, bye", "goodbye"),
    ("I'll log off now", "goodbye"),
    ("باي", "goodbye"),
    ("اشوفك بعدين", "goodbye"),
    ("خلاص لازم أمشي", "goodbye"),
    ("الى اللقاء", "goodbye"),
    ("اسلملي", "goodbye"),
    ("مع السلامة وشكرا", "goodbye"),
    ("سلام عليكم بقى", "goodbye"),
    # gratitude
    ("Thank you so much, that really helped", "gratitude"),
    ("I appreciate you listening", "gratitude"),
    ("شكرا جزيلا، ده ساعدني فعلا", "gratitude"),
    ("ممتن لك", "gratitude"),
    ("thanks", "gratitude"),
    ("thank you very much", "gratitude"),
    ("this was really helpful, thanks", "gratitude"),
    ("I'm grateful for your support", "gratitude"),
    ("thx", "gratitude"),
    ("thanks for being there", "gratitude"),
    ("appreciate it", "gratitude"),
    ("that means a lot, thank you", "gratitude"),
    ("شكرا", "gratitude"),
    ("شكرا ليكي", "gratitude"),
    ("بجد ساعدتيني كتير", "gratitude"),
    ("تسلم ايدك", "gratitude"),
    ("ربنا يخليكي", "gratitude"),
    ("ميرسي", "gratitude"),
    ("ممتن جدا", "gratitude"),
    # asking_mental_health_question
    ("I've been feeling really anxious lately and can't sleep", "asking_mental_health_question"),
    ("How do I stop overthinking everything?", "asking_mental_health_question"),
    ("حاسس بضغط نفسي شديد من الشغل", "asking_mental_health_question"),
    ("كيف أتعامل مع الاكتئاب؟", "asking_mental_health_question"),
    ("I feel so empty inside lately", "asking_mental_health_question"),
    ("can't stop crying for no reason", "asking_mental_health_question"),
    ("how do I deal with panic attacks?", "asking_mental_health_question"),
    ("my anxiety is getting worse", "asking_mental_health_question"),
    ("I'm so stressed about work it's affecting my health", "asking_mental_health_question"),
    ("I think I'm depressed", "asking_mental_health_question"),
    ("I can't sleep at night", "asking_mental_health_question"),
    ("my relationship with my mom is destroying me", "asking_mental_health_question"),
    ("I keep having intrusive thoughts", "asking_mental_health_question"),
    ("how do you cope with loss?", "asking_mental_health_question"),
    ("I'm afraid of failure", "asking_mental_health_question"),
    ("I feel lonely all the time", "asking_mental_health_question"),
    ("what's wrong with me? I have no motivation", "asking_mental_health_question"),
    ("I overthink every conversation", "asking_mental_health_question"),
    ("I'm scared to talk about my feelings with anyone", "asking_mental_health_question"),
    ("حاسس إني مش قادر أكمل", "asking_mental_health_question"),
    ("عندي قلق دايما", "asking_mental_health_question"),
    ("بحس بضيقة في صدري لما بفكر في الشغل", "asking_mental_health_question"),
    ("أمي بتضغط عليا كتير وأنا تعبان", "asking_mental_health_question"),
    ("مش بنام كويس من اسبوع", "asking_mental_health_question"),
    ("بكره نفسي", "asking_mental_health_question"),
    ("حاسس بفراغ غريب", "asking_mental_health_question"),
    ("كيف اتخلص من الافكار السلبية؟", "asking_mental_health_question"),
    ("تعبت من الحياة بصراحة", "asking_mental_health_question"),
    ("بفقد الشغف في كل حاجة", "asking_mental_health_question"),
    ("خايف من المستقبل", "asking_mental_health_question"),
    ("محدش بيفهمني", "asking_mental_health_question"),
    ("عندي نوبات هلع", "asking_mental_health_question"),
    ("ليه دايما حزين بدون سبب؟", "asking_mental_health_question"),
    ("حاسس بضغوط من كل جانب", "asking_mental_health_question"),
    # out_of_scope
    ("What's the weather in Cairo today?", "out_of_scope"),
    ("Write me a Python function to sort a list", "out_of_scope"),
    ("كم سعر الذهب اليوم؟", "out_of_scope"),
    ("اشرح لي قانون نيوتن الأول", "out_of_scope"),
    ("What's the capital of France?", "out_of_scope"),
    ("Write me a sql query", "out_of_scope"),
    ("explain quantum physics", "out_of_scope"),
    ("what time is it in tokyo", "out_of_scope"),
    ("who won the world cup", "out_of_scope"),
    ("recommend a restaurant in dubai", "out_of_scope"),
    ("translate 'hello' to spanish", "out_of_scope"),
    ("what's the weather forecast", "out_of_scope"),
    ("can you write me a poem about cars", "out_of_scope"),
    ("I'm tired of this software bug", "out_of_scope"),
    ("convert 100 USD to EUR", "out_of_scope"),
    ("what stocks should I buy?", "out_of_scope"),
    ("ما هي عاصمة استراليا؟", "out_of_scope"),
    ("اعطيني وصفة طبخ كشري", "out_of_scope"),
    ("كم باقي على رمضان", "out_of_scope"),
    ("اشرح لي خوارزمية الفقاعة", "out_of_scope"),
    ("ايه أفضل لاب توب أشتريه؟", "out_of_scope"),
    ("حول 50 ريال للجنيه المصري", "out_of_scope"),
    ("اخترعها مين الكهرباء؟", "out_of_scope"),
    ("اكتبلي مقال عن السياحة في مصر", "out_of_scope"),
]

import pandas as pd

# Pin dtype="object" (Python str), NOT pyarrow: pandas>=2.2 infer_string routes .str
# through pyarrow's RE2 regex, which rejects the \uXXXX escapes Python's re accepts.
df = pd.DataFrame(LABELED, columns=["text", "intent"]).astype({"text": "object"})
print(f"total labeled: {len(df)}")
print(df["intent"].value_counts())


# Detect Arabic via Python-side iteration (sidesteps the pyarrow regex limit). Arabic block U+0600-U+06FF.
def _has_arabic(s: str) -> bool:
    return any("؀" <= ch <= "ۿ" for ch in s)


ar_mask = df["text"].map(_has_arabic)
print(f"\nlanguage: ar={ar_mask.sum()}, en={(~ar_mask).sum()}")

## 4. Stratified train / dev / test split

Stratify by intent so every split has all 5 classes; `train_df` supplies the few-shot demos and `test_df` is the held-out eval set.

In [ ]:
from sklearn.model_selection import train_test_split

# 60/20/20 stratified by intent so every split keeps all 5 classes.
train_df, temp_df = train_test_split(df, test_size=0.40, stratify=df["intent"], random_state=SEED)
dev_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["intent"], random_state=SEED)
print(f"train={len(train_df)}  dev={len(dev_df)}  test={len(test_df)}")
print("test intent dist:", test_df["intent"].value_counts().to_dict())

## 5. The few-shot prompt (hand-written)

A single system prompt: the 5 intent definitions + 3 demos per intent (from **train** only, no test leakage), then the message to classify. We parse the label from the response (last intent mentioned wins; default `out_of_scope`).

In [ ]:
import numpy as np

# 3 demos per intent from the TRAIN split only (no test leakage).
DEMOS = []
_rng = np.random.RandomState(SEED)
for _it in INTENTS:
    _pool = train_df[train_df["intent"] == _it]
    for _, _r in _pool.sample(n=min(3, len(_pool)), random_state=_rng).iterrows():
        DEMOS.append((_r["text"], _it))

_DEFS = "\n".join(f"- {it}: {INTENT_DESCRIPTIONS[it]}" for it in INTENTS)
_DEMO_BLOCK = "\n".join(f"Message: {t}\nIntent: {it}" for t, it in DEMOS)
SYSTEM_PROMPT = (
    "You classify a user's chat message to Sakina, a multilingual mental-health support "
    "chatbot, into EXACTLY ONE intent. Messages may arrive in ANY language. Reply with ONLY "
    "the intent label (one of the five below), nothing else.\n\n"
    f"Intents:\n{_DEFS}\n\nExamples:\n{_DEMO_BLOCK}"
)
USER_TEMPLATE = "Message: {message}\nIntent:"


def classify_intent(message: str) -> str:
    """Few-shot intent classification via the LLM. Returns one of the 5 intents."""
    RATE_LIMITER.acquire()
    try:
        r = client.chat.completions.create(
            model=INTENT_MODEL, temperature=0.0, max_tokens=INTENT_MAX_TOKENS,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": USER_TEMPLATE.format(message=message)}],
        )
        out = (r.choices[0].message.content or "").lower()
        # Parse: the intent label that appears LAST wins; default out_of_scope.
        hits = [(out.rfind(it), it) for it in INTENTS if it in out]
        return max(hits)[1] if hits else "out_of_scope"
    except Exception as e:
        print("  classify err:", str(e)[:60])
        return "out_of_scope"


print(f"system prompt: {len(SYSTEM_PROMPT)} chars, {len(DEMOS)} demos\n")
for q in ["Hi", "I feel hopeless", "thanks!", "what's 2+2?", "شكرا على المساعدة"]:
    print(f"  [{classify_intent(q)}]  {q!r}")

## 6. Evaluate on the held-out test split

Plain few-shot vs measured alternatives: **zero-shot = 1.0000, DSPy MIPROv2 = 0.9583**.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

test_gold = list(test_df["intent"])
test_pred = [classify_intent(t) for t in test_df["text"]]
test_acc = accuracy_score(test_gold, test_pred)
print(f"TEST accuracy (n={len(test_gold)}): {test_acc:.4f}")
print("(plain few-shot; zero-shot=1.0000, DSPy MIPROv2=0.9583)\n")
print(classification_report(test_gold, test_pred, labels=INTENTS, digits=4, zero_division=0))

### Multilingual probe — all 20 languages the lang detector supports

The train/dev/test splits are EN + AR by design, so we separately **measure** intent accuracy across the 18 non-EN/AR languages the lang-id model supports (`bg, de, el, es, fr, hi, it, ja, nl, pl, pt, ru, sw, th, tr, ur, vi, zh`) — 5 examples per intent per language ≈ 450 rows. Lightning AI generates each example with its gold intent baked in (hand-labeling 450 rows across languages I don't speak would be error-prone), and results are cached so re-runs are instant.

In [ ]:
import json
from openai import OpenAI

# The 18 non-EN/AR languages from the lang-id artifact, so the probe covers
# everything the production language router can output.
PROBE_LANGUAGES = {
    "bg": "Bulgarian",
    "de": "German",
    "el": "Greek",
    "es": "Spanish",
    "fr": "French",
    "hi": "Hindi",
    "it": "Italian",
    "ja": "Japanese",
    "nl": "Dutch",
    "pl": "Polish",
    "pt": "Portuguese",
    "ru": "Russian",
    "sw": "Swahili",
    "th": "Thai",
    "tr": "Turkish",
    "ur": "Urdu",
    "vi": "Vietnamese",
    "zh": "Chinese (Simplified)",
}
N_PER_INTENT_PER_LANG = 5  # 5 per (intent, language) -> 25 per language -> 450 total

PROBE_CACHE = ARTIFACTS_DIR / "intent_multilingual_probe.json"

PROBE_PROMPT = """Generate {n} short chat messages in {language_name} that express the intent: {intent} ({intent_desc}).

Rules:
- Each message should be natural, like a real user typing to a chatbot.
- Vary vocabulary, situations, and length (5-25 words).
- Do NOT include the intent label in the message itself.
- Use natural {language_name} — including informal forms or common dialects if appropriate.
- Output ONLY a JSON array of {n} strings. No prose, no markdown fences, no labels."""


def _generate_probe_for(lang_code: str, lang_name: str, intent: str, n: int = 5) -> list[str]:
    prompt = PROBE_PROMPT.format(
        n=n,
        language_name=lang_name,
        intent=intent,
        intent_desc=INTENT_DESCRIPTIONS[intent],
    )
    for attempt in range(3):
        RATE_LIMITER.acquire()
        try:
            client = OpenAI(base_url=LIGHTNING_BASE_URL, api_key=LIGHTNING_API_KEY)
            response = client.chat.completions.create(
                model=LIGHTNING_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,  # higher T for linguistic variety
                max_tokens=600,
            )
            raw = response.choices[0].message.content.strip()
            # Strip code fences the model sometimes adds despite instructions.
            import re as _re

            raw = _re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=_re.MULTILINE).strip()
            arr = json.loads(raw)
            if isinstance(arr, list) and len(arr) == n:
                return [str(s) for s in arr]
            print(
                f"  [warn] {lang_code}/{intent}: got {len(arr) if isinstance(arr, list) else 'non-list'}, retrying ({attempt + 1}/3)"
            )
        except Exception as e:
            print(
                f"  [warn] {lang_code}/{intent} attempt {attempt + 1}: {type(e).__name__}: {str(e)[:80]}"
            )
            time.sleep(2)
    print(f"  [error] {lang_code}/{intent} failed all retries")
    return []


# Load or generate the probe set (cached so re-runs are instant).
if PROBE_CACHE.exists():
    with open(PROBE_CACHE, encoding="utf-8") as f:
        probe_rows = json.load(f)
    print(f"loaded {len(probe_rows)} probe rows from cache: {PROBE_CACHE.name}")
else:
    print(
        f"generating {len(PROBE_LANGUAGES)} languages × {len(INTENTS)} intents × {N_PER_INTENT_PER_LANG} = {len(PROBE_LANGUAGES) * len(INTENTS) * N_PER_INTENT_PER_LANG} probe rows"
    )
    print(f"est. wall time: {len(PROBE_LANGUAGES) * len(INTENTS) * 60 / (15 * 0.9) / 60:.1f} min\n")
    probe_rows = []
    for lang_code, lang_name in PROBE_LANGUAGES.items():
        for intent in INTENTS:
            texts = _generate_probe_for(lang_code, lang_name, intent, N_PER_INTENT_PER_LANG)
            for t in texts:
                probe_rows.append({"text": t, "gold": intent, "lang": lang_code})
        print(
            f"  {lang_code} ({lang_name}): {sum(1 for r in probe_rows if r['lang'] == lang_code)} rows so far"
        )
    with open(PROBE_CACHE, "w", encoding="utf-8") as f:
        json.dump(probe_rows, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] cached probe at {PROBE_CACHE}")

probe_df = pd.DataFrame(probe_rows)
print(f"\nprobe size: {len(probe_df)} rows across {probe_df['lang'].nunique()} languages")
print("coverage per (lang, intent):")
print(probe_df.groupby(["lang", "gold"]).size().unstack(fill_value=0))

## 7. Score the multilingual probe (the real 20-language validation)

Classify all 450 probe rows — **~33 min** on the 15 req/min cap.

In [ ]:
from sklearn.metrics import accuracy_score

probe_gold = [r["gold"] for r in probe_rows]
probe_pred = [classify_intent(r["text"]) for r in probe_rows]
probe_acc = accuracy_score(probe_gold, probe_pred)
probe_df["pred"] = probe_pred
probe_df["correct"] = probe_df["gold"] == probe_df["pred"]

print(f"20-language probe accuracy (n={len(probe_gold)}): {probe_acc:.4f}\n")
print("per-language:")
print(probe_df.groupby("lang")["correct"].mean().round(3).to_string())
print("\nper-intent:")
print(probe_df.groupby("gold")["correct"].mean().round(3).to_string())
print("\nNote: 'greeting' is lowest because many synthetic greeting rows are COMPOUND "
      "(a greeting + an off-topic request); the model reasonably labels those out_of_scope, "
      "so true quality is higher than the headline number.")

## 8. Export `intent_few_shot.json`

The API replays this exact prompt with a plain `openai` call — **no `dspy` dependency**.

In [ ]:
import json

artifact = {
    "version": "2.0",
    "approach": "few-shot LLM prompting (DSPy/MIPROv2 removed — it scored below zero-shot)",
    "model": INTENT_MODEL,
    "max_tokens": INTENT_MAX_TOKENS,
    "temperature": 0.0,
    "intents": INTENTS,
    "intent_descriptions": INTENT_DESCRIPTIONS,
    "system_prompt": SYSTEM_PROMPT,
    "user_template": USER_TEMPLATE,
    "parse": "lowercase the response; the intent label that appears LAST wins; default out_of_scope",
    "eval": {
        "test_n": len(test_gold),
        "test_accuracy": round(float(test_acc), 4),
        "multilingual_probe_n": len(probe_gold),
        "multilingual_probe_accuracy": round(float(probe_acc), 4),
        "probe_per_language": {k: round(float(v), 3) for k, v in probe_df.groupby("lang")["correct"].mean().items()},
        "probe_per_intent": {k: round(float(v), 3) for k, v in probe_df.groupby("gold")["correct"].mean().items()},
        "baselines_on_test": {"zero_shot": 1.0, "dspy_miprov2": 0.9583},
        "note": "Probe = 450 synthetic messages (18 non-EN/AR langs x 5 intents x 5); gold = generation intent. 'greeting' is low due to compound synthetic rows, not a real model failure.",
    },
}
out_path = ARTIFACTS_DIR / "intent_few_shot.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)
print(f"[OK] saved {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")
print("model:", INTENT_MODEL, "| test:", artifact["eval"]["test_accuracy"],
      "| probe:", artifact["eval"]["multilingual_probe_accuracy"])

## 9. Reload + verify the runtime path

In [ ]:
import json

loaded = json.load(open(ARTIFACTS_DIR / "intent_few_shot.json", encoding="utf-8"))
print("approach :", loaded["approach"])
print("model    :", loaded["model"])
print("test/probe acc:", loaded["eval"]["test_accuracy"], "/", loaded["eval"]["multilingual_probe_accuracy"])


def api_classify(message: str) -> str:
    """Exactly what the API's intent module will do — plain call, no dspy."""
    RATE_LIMITER.acquire()
    r = client.chat.completions.create(
        model=loaded["model"], temperature=loaded["temperature"], max_tokens=loaded["max_tokens"],
        messages=[{"role": "system", "content": loaded["system_prompt"]},
                  {"role": "user", "content": loaded["user_template"].format(message=message)}],
    )
    out = (r.choices[0].message.content or "").lower()
    hits = [(out.rfind(it), it) for it in loaded["intents"] if it in out]
    return max(hits)[1] if hits else "out_of_scope"


print("\nsmoke (multilingual):")
for q in ["good evening", "I can't sleep and feel anxious", "bye for now", "write me python code", "merci beaucoup"]:
    print(f"  [{api_classify(q)}]  {q!r}")